In [0]:
from pyspark.sql import functions as F

files_df = spark.table(
    "nyc_citi_bike_catalog.audit.audit_files_tbl"
)

events_df = spark.table(
    "nyc_citi_bike_catalog.audit.audit_events_tbl"
)

In [0]:
extracted_df = (
    events_df
    .filter(
        (F.col("stage") == "CSV_EXTRACT") &
        (F.col("status") == "SUCCESS")
    )
    .select("source_key")
    .distinct()
)

In [0]:
bronze_completed_df = (
    events_df
    .filter(
        (F.col("stage") == "BRONZE") &
        (F.col("status") == "SUCCESS")
    )
    .select("source_key")
    .distinct()
)

In [0]:
bronze_completed_df.display()

In [0]:
pending_keys_df = extracted_df.join(
    bronze_completed_df,
    on="source_key",
    how="left_anti"
)

pending_files_df = (
    files_df
    .join(pending_keys_df, on="source_key", how="inner")
    .select(
        "source_key",
        "zip_key",
        "batch_id",
        "csv_name",
        "csv_path",
        "file_size_bytes"
    )
)

In [0]:
pending_files_df.display()

In [0]:
from bronze import *
from bronze_spark_schemas import *

In [0]:
df_batch_load = spark.read.format('csv')\
                    .option("recursiveFileLookup", "true")\
                    .schema(RIDE_SCHEMA)\
                    .option('header', True)\
                    .load('/Volumes/nyc_citi_bike_catalog/audit/data/landing/trips/')\
                    .withColumn(
                        "csv_path",
                        F.regexp_replace(F.col("_metadata.file_path"), "dbfs:", "")
                    )

In [0]:
df_batch_load.display()

In [0]:
metadata_df = (
    pending_files_df
    .select(["csv_path", "source_key", "batch_id"])
)

src_mapped_df = df_batch_load.join(
                    metadata_df, 
                    on='csv_path', 
                    how = "inner"
                ).drop(F.col('csv_path'))
src_mapped_df.display()

In [0]:
agg_src_mapped_df = src_mapped_df.groupby('source_key').agg(F.count('ride_id').alias("row_count"))

In [0]:
events_df.display()

In [0]:
src_mapped_df.write.format('delta')\
    .option('optimizeWrite', True)\
    .option('mergeSchema', "true")\
    .mode('append')\
    .saveAsTable('nyc_citi_bike_catalog.bronze.raw_trips_tbl')
